# OpenProblems — scPRINT-2 fine-tuned

This notebook exposes the held-out split, model stages, classification details, individual scIB metrics, and manuscript comparison without hiding them behind one runner.


## Comparison contract

Classification uses the frozen held-out donor per dataset. Batch integration uses the immutable four-dataset snapshot and a fixed five-metric denominator.


In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from scprint2.benchmark.openproblems import (
    DATASETS,
    OpenProblemsConfig,
    build_manifest,
    classification_scores,
    embed_all,
    finetune_openproblems_model,
    load_model,
    load_openproblems_dataset,
    select_embedding_view,
    summarize_openproblems_dataset,
)
from scprint2.plotting.openproblems import load_batch_tables, load_classification_tables


REPO_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").exists())
INPUT_DIR = REPO_ROOT / "figures/plot_results_inputs"
DETAIL = REPO_ROOT / "data/results/openproblems_validation/final_20260815/scib_all_models.csv"
RUN_MODEL = False


No module named 'adasplash'
FlashAttention is not installed, not using it..


## Dataset and held-out audit

The protocol table makes every test donor explicit before model loading.


In [2]:
protocol = pd.DataFrame([
    {"dataset": key, "source": value["name"], "held_out_donor": ", ".join(value["test_donors"])}
    for key, value in DATASETS.items()
]).set_index("dataset")
display(protocol)


,source,held_out_donor
dataset,,
dkd,cellxgene_census/dkd,control_3
gtex_v9,cellxgene_census/gtex_v9,GTEX-16BQI
hypomap,cellxgene_census/hypomap,SRR9000488
mouse_pancreas_atlas,cellxgene_census/mouse_pancreas_atlas,mouse_pancreatic_islet_atlas_Hrovatin__VSG__MU...


## MMD configuration

`mmd_batch_key` is user-defined. The default `donor_id` groups cells for MMD; the cell-type token remains the regularized representation and the default weight is `0.03`.


In [3]:
benchmark_config = OpenProblemsConfig(
    dataset="dkd", mode="finetune", input=Path("/path/to/dkd.h5ad"),
    checkpoint=Path("/path/to/small-v2.ckpt"), output_dir=Path("/path/to/results"),
    embedding_view="cell_type",
    mmd_mode="fixed", mmd_weight=0.03, mmd_batch_key="donor_id",
)
display(pd.Series(benchmark_config.as_dict(), name="scPRINT-2 configuration"))


dataset                                   dkd
mode                                 finetune
input                       /path/to/dkd.h5ad
checkpoint             /path/to/small-v2.ckpt
output_dir                   /path/to/results
embedding_view                      cell_type
num_workers                                 8
seed                                        0
mmd_mode                                fixed
mmd_weight                               0.03
mmd_batch_key                        donor_id
classification_only                     False
scib_only                               False
Name: scPRINT-2 configuration, dtype: object

## Optional fine-tuning

The expensive path is split into data loading, audit, checkpoint load, fine-tuning, embedding, classification, and view selection.


In [4]:
if RUN_MODEL:
    openproblems_data = load_openproblems_dataset(benchmark_config)
    display(summarize_openproblems_dataset(openproblems_data, benchmark_config))


In [5]:
if RUN_MODEL:
    openproblems_model = load_model(str(benchmark_config.checkpoint))
    print(type(openproblems_model).__name__)


In [6]:
if RUN_MODEL:
    openproblems_model = finetune_openproblems_model(
        openproblems_model, openproblems_data, benchmark_config
    )
    print("fine-tuning complete")


In [7]:
if RUN_MODEL:
    openproblems_embedding, embedding_blocks = embed_all(openproblems_model, openproblems_data, benchmark_config.num_workers)
    display(pd.Series({key: list(value.shape) for key, value in openproblems_embedding.obsm.items()}, name="embedding blocks"))


In [8]:
if RUN_MODEL:
    classification = classification_scores(openproblems_embedding, openproblems_model, DATASETS[benchmark_config.dataset]["test_donors"])
    display(pd.DataFrame(classification).T)


In [9]:
if RUN_MODEL:
    embedding_name, selected_blocks = select_embedding_view(
        openproblems_embedding, embedding_blocks,
        mode=benchmark_config.mode, embedding_view=benchmark_config.embedding_view,
    )
    manifest = build_manifest(
        benchmark_config.as_namespace(), spec=DATASETS[benchmark_config.dataset],
        embedded=openproblems_embedding, embedding_name=embedding_name,
        embedding_blocks=selected_blocks,
    )
    display(pd.Series(manifest, name="run manifest"))


## Classification details

The four held-out datasets are kept separate; macro-F1, weighted-F1, micro-F1, and accuracy are not collapsed into one number.


In [10]:
classification_tables = load_classification_tables(INPUT_DIR / "fig3_openproblems_classification_scores.csv")
classification = pd.DataFrame({
    metric: table.loc["scPRINT-2 (fine-tune)"]
    for metric, table in classification_tables.items()
})
display(classification)


,f1_macro,f1_weighted,accuracy,f1_micro
dataset,,,,
dkd,0.857304,0.910927,0.938613,0.938613
gtex_v9,0.784242,0.875409,0.910845,0.910845
hypomap,0.872824,0.993235,0.995438,0.995438
mouse_pancreas_atlas,0.899435,0.981093,0.987078,0.987078


In [11]:
ax = classification.plot.bar(figsize=(10, 4.5), ylim=(0, 1), rot=15)
ax.set_title("Held-out classification by dataset")
ax.set_ylabel("score")
ax.legend(title="metric", ncol=2)
plt.tight_layout()


In [12]:
classification_range = pd.DataFrame({
    "minimum": classification.min(),
    "mean": classification.mean(),
    "maximum": classification.max(),
})
display(classification_range)


,minimum,mean,maximum
f1_macro,0.784242,0.853451,0.899435
f1_weighted,0.875409,0.940166,0.993235
accuracy,0.910845,0.957993,0.995438
f1_micro,0.910845,0.957993,0.995438


## scIB breakdown

The aggregate table is followed by the raw per-metric rows used for the four datasets.


In [13]:
batch_tables = load_batch_tables(INPUT_DIR / "fig5_op_scib_scores.csv")
aggregates = pd.DataFrame({
    metric: table.loc["scPRINT-2-FT"]
    for metric, table in batch_tables.items()
})
display(aggregates)


,Total,Bio conservation,Batch correction
dkd,0.663060,0.703518,0.602372
gtex_v9,0.659950,0.804634,0.442925
hypomap,0.653981,0.645236,0.667098
mouse_pancreas_atlas,0.730544,0.840518,0.565582


In [14]:
ax = aggregates.plot.bar(figsize=(10, 4.5), ylim=(0, 1), rot=15)
ax.set_title("Batch correction, biological conservation, and total")
ax.set_ylabel("score")
plt.tight_layout()


In [15]:
raw_scib = pd.read_csv(DETAIL)
method_column = "scPRINT-2 FT cell_type MMD 0.03"
detailed_scib = raw_scib.pivot(index="metric", columns="dataset", values=method_column)
display(detailed_scib)


dataset,dkd,gtex_v9,hypomap,mouse_pancreas_atlas
metric,,,,
ari,0.913979,0.889335,4.584304e-01,0.898900
asw_batch,0.863888,0.853730,7.136738e-01,0.768098
asw_label,0.737357,0.662230,8.369819e-01,0.728870
cell_cycle_conservation,0.848108,0.740548,6.290994e-01,0.690652
clisi,1.000000,0.999744,1.000000e+00,1.000000
graph_connectivity,0.967018,0.940786,9.132413e-01,0.976376
hvg_overlap,NaN,NaN,NaN,NaN
ilisi,0.282351,0.065771,2.896234e-17,0.089581
isolated_label_asw,NaN,0.579723,1.667370e-01,0.684177


In [16]:
available = detailed_scib.notna().sum(axis=1).rename("datasets available")
display(available.to_frame())
ax = detailed_scib.mean(axis=1, skipna=True).sort_values().plot.barh(
    figsize=(8, 6), color="#4c78a8", xlim=(0, 1)
)
ax.set_title("Mean raw scIB metric across available datasets")
ax.set_xlabel("score")
plt.tight_layout()


,datasets available
metric,
ari,4
asw_batch,4
asw_label,4
cell_cycle_conservation,4
clisi,4
graph_connectivity,4
hvg_overlap,0
ilisi,4
isolated_label_asw,3


## Comparison

The final view places the selected method next to scPRINT-1, scPRINT-2 zero-shot, scPRINT-2 fine-tuned, and TranscriptFormer when those snapshot rows are available.


In [17]:
comparison_methods = [
    method for method in ["scPRINT-1", "scPRINT-2 ZS cell_type", "scPRINT-2-FT", "TranscriptFormer"]
    if method in batch_tables["Total"].index
]
comparison = batch_tables["Total"].loc[comparison_methods]
display(comparison)
ax = comparison.mean(axis=1).sort_values().plot.barh(figsize=(8, 4), color="#59a14f", xlim=(0, 1))
ax.set_title("Mean OpenProblems total across four datasets")
ax.set_xlabel("score")
plt.tight_layout()


,dkd,gtex_v9,hypomap,mouse_pancreas_atlas
scPRINT-1,0.411256,0.581105,0.553168,0.594673
scPRINT-2 ZS cell_type,0.535064,0.585228,0.598376,0.621788
scPRINT-2-FT,0.663060,0.659950,0.653981,0.730544
TranscriptFormer,0.506691,0.589030,0.535379,0.535864


## Interpretation boundary

Missing metric values remain unavailable evidence. The plotted totals use `0.4 × Batch correction + 0.6 × Bio conservation`; the two manuscript visualization policies are handled separately in `figures/plot_results.ipynb`.
